# ViT team starter: k=1 real/fake detection

**Our first goal:** train a small classifier using a pretrained CLIP ViT-L/14 image encoder.
CLIP turns each image into a feature vector; we freeze CLIP and train only one linear layer.
No text prompts are needed. This is a supervised classifier, not zero-shot CLIP.

We start with Medini's `holdout_ADM_k1_rep0` CSVs. The earlier index listed
**Midjourney for training and ADM for testing**; the notebook prints the actual generators
from your current CSVs so you can confirm them.

**Use:** upload this notebook to Google Colab, select a GPU runtime, and run cells in order.
Add a shortcut to the shared dataset in My Drive named `GenImage_Compact_shared`.
The images and CSVs are read only. Results go into a separate `ViT_team_runs` folder.

| Teammate | Start here | Deliverable |
|---|---|---|
| A: data / CARC | Sections 1–3 | CSVs and an image batch load |
| B: model / training | Sections 4–5 | Saved classifier and training history |
| C: evaluation / CNN | Sections 6–7 | Metrics and per-image predictions |

Work from one shared master notebook; use different run folders when running separately.
This notebook has **not been trained on your data yet**; no example performance is presented as a result.


## 1. Setup
In Colab choose **Runtime → Change runtime type → GPU** first.
This cell uses Colab's existing PyTorch and installs the original CLIP implementation
and small supporting packages. On CARC, prepare your environment first and run inside
an allocated GPU job; skip the Colab installation branch.


In [2]:
import sys
import importlib

print("Python environment:", sys.executable)

for name in ["torch", "torchvision", "clip", "pandas",
             "numpy", "sklearn", "matplotlib", "PIL", "tqdm"]:
    try:
        importlib.import_module(name)
        print(f"✓ {name}: works")
    except Exception as error:
        print(f"✗ {name}: {error}")

try:
    import torch
    print("GPU available:", torch.cuda.is_available())
except ImportError:
    pass

Python environment: /apps/conda/envs/ood-jupyterlab-4.6/bin/python
✗ torch: No module named 'torch'
✗ torchvision: No module named 'torchvision'
✗ clip: No module named 'clip'
✗ pandas: No module named 'pandas'
✗ numpy: No module named 'numpy'
✗ sklearn: No module named 'sklearn'
✗ matplotlib: No module named 'matplotlib'
✗ PIL: No module named 'PIL'
✗ tqdm: No module named 'tqdm'


In [1]:
import importlib.util
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/openai/CLIP.git",
        "pandas", "scikit-learn", "matplotlib", "tqdm"
    ])
else:
    print("Use an environment with torch, torchvision, CLIP, pandas, scikit-learn, matplotlib and tqdm.")


ModuleNotFoundError: No module named 'google'

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import copy
import hashlib
import importlib.metadata
import json
import random
import re
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.nn import functional as F
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
from tqdm.auto import tqdm
import clip

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU works for a small check, but use a GPU for the full CLIP feature extraction.")


## 2. Settings — teammate A
Usually only `DATA_ROOT` needs changing. This is the folder containing `ADM`,
`Midjourney`, and the other generators—not the Google Drive web URL.

On CARC, set `DATA_ROOT` to the transferred dataset directory and `OUTPUT_BASE`
to your own project directory. You do not need to mount Drive there.


In [ ]:
DATA_ROOT = Path("data/extracted/GenImage_Compact").resolve()

EXPERIMENT = "holdout_ADM_k2_rep0"
MANIFEST_DIR = DATA_ROOT / "_experiment_manifests" / EXPERIMENT
OUTPUT_BASE = Path("/content/drive/MyDrive/ViT_team_runs") if IN_COLAB else Path.cwd() / "ViT_team_runs"

MODEL_NAME = "ViT-L/14"
SEED = 17
IMAGE_BATCH_SIZE = 16  # Reduce to 8 if GPU memory runs out.
HEAD_BATCH_SIZE = 128
MAX_EPOCHS = 20
PATIENCE = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
THRESHOLD = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

for split in ["train", "val", "test"]:
    if not (MANIFEST_DIR / f"{split}.csv").is_file():
        raise FileNotFoundError(f"Missing {split}.csv in {MANIFEST_DIR}. Check DATA_ROOT and the Drive shortcut.")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = OUTPUT_BASE / f"{EXPERIMENT}_seed{SEED}_{stamp}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Read CSVs from:", MANIFEST_DIR)
print("Save results to:", RUN_DIR)


## 3. Load the CSVs and check one image batch — teammate A
Labels follow Medini's notebook: **real = 0, fake = 1**.
Paths saved under another person's Drive shortcut are automatically mapped to `DATA_ROOT`.
We use the provided splits directly; we do not create new splits here.


In [ ]:
def resolve_image_path(raw_path):
    text = str(raw_path).replace("\\", "/")
    # Prefer the configured data root when the CSV contains a Colab absolute path.
    marker = re.search(r"(?:^|/)GenImage_Compact(?:_shared)?/", text)
    if marker:
        return DATA_ROOT / text[marker.end():]
    path = Path(text)
    return path if path.is_absolute() else DATA_ROOT / path


def read_manifest(split):
    df = pd.read_csv(MANIFEST_DIR / f"{split}.csv")
    required = {"path", "label", "source_generator"}
    if not required.issubset(df.columns):
        raise ValueError(f"{split}: missing columns {required - set(df.columns)}")
    if df["path"].isna().any() or not df["label"].isin([0, 1]).all():
        raise ValueError(f"{split}: paths must exist in the CSV and labels must be 0 or 1.")
    df["label"] = df["label"].astype(int)
    if set(df["label"]) != {0, 1}:
        raise ValueError(f"{split}: both real and fake examples are required.")
    df["resolved_path"] = df["path"].map(lambda p: str(resolve_image_path(p)))
    missing = [p for p in df["resolved_path"] if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError(f"{split}: {len(missing)} missing images. First: {missing[0]}")
    return df


frames = {split: read_manifest(split) for split in ["train", "val", "test"]}
train_gens = set(frames["train"].query("label == 1")["source_generator"])
val_gens = set(frames["val"].query("label == 1")["source_generator"])
test_gens = set(frames["test"].query("label == 1")["source_generator"])
assert len(train_gens) == 2, f"Expected k=2; found {train_gens}"
assert train_gens == val_gens, "Train and validation fake generators differ."
assert train_gens.isdisjoint(test_gens), "The test generator must be unseen during training."
assert test_gens == {"ADM"}, f"Expected ADM holdout; found {test_gens}"

for split, df in frames.items():
    print(split, "| real:", int((df.label == 0).sum()), "| fake:", int((df.label == 1).sum()))
print("Train fake generator:", sorted(train_gens))
print("Test fake generator:", sorted(test_gens))

# Lightweight overlap check; this does not redo Medini's image-content audit.
for column in ["resolved_path", "sha256"]:
    if all(column in df.columns for df in frames.values()):
        values = pd.concat([df[column] for df in frames.values()], ignore_index=True)
        assert values.notna().all() and values.is_unique, f"Duplicate or missing {column} across the CSVs."

# Save copies of the exact input manifests and checksums for this run.
manifest_checksums = {}
for split in frames:
    raw = (MANIFEST_DIR / f"{split}.csv").read_bytes()
    manifest_checksums[split] = hashlib.sha256(raw).hexdigest()
    (RUN_DIR / f"input_{split}.csv").write_bytes(raw)
display(frames["train"][["path", "label", "source_generator"]].head())


## 4. Load frozen CLIP and extract image features — teammate B
The first model load downloads the pretrained weights. Use internet access for this setup step.
CLIP provides its own resize/crop/normalization, so do not add ImageNet normalization.
The encoder stays frozen and in evaluation mode. Features are held in memory for this run.


In [ ]:
encoder, preprocess = clip.load(MODEL_NAME, device=DEVICE, jit=False)
encoder.eval()
for parameter in encoder.parameters():
    parameter.requires_grad_(False)
print("Loaded:", MODEL_NAME)
print("Preprocessing:", preprocess)


class ManifestImages(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        try:
            with Image.open(row.resolved_path) as image:
                tensor = preprocess(image.convert("RGB"))
        except Exception as error:
            raise RuntimeError(f"Could not decode image: {row.resolved_path}") from error
        return tensor, int(row.label)


def image_loader(frame):
    return DataLoader(ManifestImages(frame), batch_size=IMAGE_BATCH_SIZE,
                      shuffle=False, num_workers=0, pin_memory=DEVICE.type == "cuda")


batch_images, batch_labels = next(iter(image_loader(frames["train"])))
print("Image batch:", tuple(batch_images.shape))
print("Labels:", batch_labels.tolist())
assert batch_images.shape[1:] == (3, 224, 224)


In [ ]:
@torch.no_grad()
def extract_features(frame, description):
    features, labels = [], []
    encoder.eval()
    for images, target in tqdm(image_loader(frame), desc=description):
        encoded = encoder.encode_image(images.to(DEVICE)).float()
        encoded = F.normalize(encoded, dim=-1)
        if not torch.isfinite(encoded).all():
            raise RuntimeError("Nonfinite CLIP features encountered.")
        features.append(encoded.cpu())
        labels.append(target)
    return torch.cat(features), torch.cat(labels).float()


X_train, y_train = extract_features(frames["train"], "Training features")
X_val, y_val = extract_features(frames["val"], "Validation features")
print("Train features:", tuple(X_train.shape))
print("Validation features:", tuple(X_val.shape))
# Test image features are extracted only in the final evaluation section.


## 5. Train the small classifier — teammate B
Only the linear head learns. Validation AUROC selects the checkpoint; test results
are not used to choose an epoch. These settings are a simple starting point.


In [ ]:
def classification_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= THRESHOLD).astype(int)
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision_fake": float(precision_score(labels, predictions, zero_division=0)),
        "recall_fake": float(recall_score(labels, predictions, zero_division=0)),
        "f1_fake": float(f1_score(labels, predictions, zero_division=0)),
        "auroc": float(roc_auc_score(labels, probabilities)),
    }


@torch.no_grad()
def predict_features(head, features):
    head.eval()
    outputs = [torch.sigmoid(head(batch.to(DEVICE)).squeeze(-1)).cpu()
               for batch in features.split(HEAD_BATCH_SIZE)]
    return torch.cat(outputs).numpy()


# Reset seeds here so re-running training starts the head reproducibly.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
head = nn.Linear(X_train.shape[1], 1).to(DEVICE)
optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
loss_fn = nn.BCEWithLogitsLoss()
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=HEAD_BATCH_SIZE,
                          shuffle=True, generator=torch.Generator().manual_seed(SEED))

best_auc, best_val_loss = -1.0, float("inf")
best_state, best_epoch, stale_epochs = None, None, 0
history = []
for epoch in range(1, MAX_EPOCHS + 1):
    head.train()
    total_loss = 0.0
    for features, labels in train_loader:
        features, labels = features.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(head(features).squeeze(-1), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    val_prob = predict_features(head, X_val)
    metrics = classification_metrics(y_val.numpy(), val_prob)
    # BCE from probabilities is used only for logging/tie-breaking.
    clipped = np.clip(val_prob.astype(np.float64), 1e-7, 1 - 1e-7)
    truth = y_val.numpy()
    val_loss = float(-np.mean(truth * np.log(clipped) + (1 - truth) * np.log1p(-clipped)))
    history.append({"epoch": epoch, "train_loss": total_loss / len(y_train),
                    "val_loss": val_loss, **{f"val_{k}": v for k, v in metrics.items()}})
    print(f"Epoch {epoch:02d} | train loss {history[-1]['train_loss']:.4f} | "
          f"val AUROC {metrics['auroc']:.4f} | val F1 {metrics['f1_fake']:.4f}")

    improved = (metrics["auroc"] > best_auc or
                (metrics["auroc"] == best_auc and val_loss < best_val_loss))
    if improved:
        best_auc, best_val_loss = metrics["auroc"], val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        best_epoch, stale_epochs = epoch, 0
    else:
        stale_epochs += 1
    if stale_epochs >= PATIENCE:
        print("Early stopping.")
        break

assert best_state is not None
head.load_state_dict(best_state)
head.eval()
history_df = pd.DataFrame(history)
history_df.to_csv(RUN_DIR / "training_history.csv", index=False)
print("Restored best validation checkpoint from epoch", best_epoch)


In [ ]:
config = {
    "experiment": EXPERIMENT, "model": MODEL_NAME, "seed": SEED,
    "backbone_frozen": True, "features_l2_normalized": True,
    "feature_dim": int(X_train.shape[1]), "augmentation": "none",
    "label_mapping": {"real": 0, "fake": 1}, "threshold": THRESHOLD,
    "train_generators": sorted(train_gens), "test_generators": sorted(test_gens),
    "counts": {split: len(frame) for split, frame in frames.items()},
    "manifest_sha256": manifest_checksums, "preprocessing": repr(preprocess),
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "head_batch_size": HEAD_BATCH_SIZE, "image_batch_size": IMAGE_BATCH_SIZE,
    "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "best_epoch": best_epoch,
    "selection_metric": "validation AUROC, then validation BCE",
    "created_utc": stamp, "python": platform.python_version(), "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
}
for package in ["torchvision", "pandas", "scikit-learn", "numpy", "clip"]:
    try:
        config[package + "_version"] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        pass
try:
    config["clip_install_source"] = json.loads(importlib.metadata.distribution("clip").read_text("direct_url.json") or "{}")
except (ValueError, importlib.metadata.PackageNotFoundError):
    pass

(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
torch.save({"head_state_dict": best_state, "config": config}, RUN_DIR / "best_head.pt")
print("Saved classifier and settings:", RUN_DIR)


## 6. Validation results — teammate C
Check these first. A score near chance is a valid observation, not a reason to inspect
test predictions and keep changing settings. Agree on changes using validation results.


In [ ]:
def save_predictions(split, frame, labels, probabilities):
    result = frame.drop(columns=["resolved_path"]).copy()
    # The source path is retained as an identifier the CNN team can also match.
    if "image_id" not in result.columns:
        result["image_id"] = result["sha256"] if "sha256" in result.columns else result["path"]
    result["prob_fake"] = probabilities
    result["pred_label"] = (probabilities >= THRESHOLD).astype(int)
    result["experiment"] = EXPERIMENT
    result["seed"] = SEED
    result["threshold"] = THRESHOLD
    result.to_csv(RUN_DIR / f"{split}_predictions.csv", index=False)
    metrics = classification_metrics(labels, probabilities)
    (RUN_DIR / f"{split}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    return metrics


val_prob = predict_features(head, X_val)
validation_metrics = save_predictions("val", frames["val"], y_val.numpy(), val_prob)
display(pd.DataFrame([validation_metrics], index=["validation"]).round(4))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history_df.epoch, history_df.train_loss, label="train")
axes[0].plot(history_df.epoch, history_df.val_loss, label="validation")
axes[0].set(xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[1].plot(history_df.epoch, history_df.val_auroc)
axes[1].set(xlabel="Epoch", ylabel="Validation AUROC", ylim=(0, 1))
fig.tight_layout()
fig.savefig(RUN_DIR / "training_curves.png", dpi=150)
plt.show()


## 7. Final held-out ADM test — teammate C
When the baseline configuration is settled, change `RUN_FINAL_TEST` to `True` below.
This scores the selected checkpoint on ADM, which was excluded from training.
The test CSV's labels/paths were checked earlier, but its images have not been encoded
or scored until now. Use these results for reporting, not for choosing more settings.


In [ ]:
RUN_FINAL_TEST = False  # Set True when ready to report the baseline.

if RUN_FINAL_TEST:
    X_test, y_test = extract_features(frames["test"], "Held-out test features")
    test_prob = predict_features(head, X_test)
    test_metrics = save_predictions("test", frames["test"], y_test.numpy(), test_prob)
    display(pd.DataFrame([test_metrics], index=["held-out ADM test"]).round(4))
    predicted = (test_prob >= THRESHOLD).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test.numpy().astype(int), predicted,
                                            labels=[0, 1], display_labels=["Real", "Fake"])
    plt.title("Held-out ADM test")
    plt.tight_layout()
    plt.savefig(RUN_DIR / "test_confusion_matrix.png", dpi=150)
    plt.show()
else:
    print("Test evaluation skipped. Set RUN_FINAL_TEST=True when the baseline is settled.")

print("Results folder:", RUN_DIR)
print("Files:", sorted(p.name for p in RUN_DIR.iterdir()))


## What to share and what to do next

Share `config.json`, `training_history.csv`, `val_metrics.json`, `best_head.pt`, and,
after final evaluation, `test_metrics.json` and `test_predictions.csv`.
The checkpoint stores the small head; reloading it also requires the same CLIP backbone
and preprocessing recorded in the config.

Ask the CNN team to use the **same CSVs**, labels, threshold and metric definitions.
F1/precision/recall here refer to the **fake class**. AUROC uses probabilities, not hard predictions.

Once k=1 works, make a new run by changing `EXPERIMENT` to the matching k=2,4,7 folder
and updating the k=1 assertion in Section 3 to match that experiment. Keep the rest of
the baseline unchanged. Try additional seeds separately. Leave augmentation and
fine-tuning for a later notebook revision.

### Team notes
- A — dataset / CARC notes:
- B — training notes:
- C — evaluation / CNN notes:

### Common fixes
- **Missing CSVs:** check the dataset shortcut, `DATA_ROOT`, and experiment folder.
- **Missing image paths:** verify the dataset was mounted/transferred, not just the CSVs.
- **GPU memory error:** reduce `IMAGE_BATCH_SIZE` to 8 or 4 and restart the runtime.
- **Slow Drive reads:** feature extraction reads images once per split; on CARC use a local project copy.
- **Disconnected Colab runtime:** saved run outputs remain in your own Drive folder.

References: [OpenAI CLIP code and preprocessing](https://github.com/openai/CLIP),
[frozen CLIP fake-image detection baseline](https://github.com/WisconsinAIVision/UniversalFakeDetect).
This notebook trains its own classifier; it does not load that detector's trained head.
